<a href="https://colab.research.google.com/github/Awuor99/SQL-Data-Cleaning-Analysis-Project/blob/main/Cafe_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [147]:
#Loading SQL Extension
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [148]:
#Connecting to SQLite
%sql sqlite://

In [149]:
#Importing pandas Library
import pandas as pd

In [150]:
# loading our file
df = pd.read_csv("/dirty_cafe_sales.csv")

In [151]:
#Loading sqlite3 library
#Establishing connection to our database
import sqlite3

conn = sqlite3.connect(':memory:')

df.to_sql('dirt_cafe_sales', conn, if_exists='replace', index=False)

10000

In [152]:
#We want to see the first 5 rows of our data
pd.read_sql("SELECT * FROM dirt_cafe_sales LIMIT 5;", conn)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,9/8/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,5/16/2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,7/19/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,4/27/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,6/11/2023


In [153]:
#Changing column names i.e Transaction ID, Price Per Unit, Total Spent, Payment Method, Transaction Date
#Transaction ID AS Transaction_ID,
#Price Per Unit AS Price_Per_Unit,
#Total Spent AS Total_Spent,
#Payment Method AS Payment_Method,
#Transaction Date AS Transaction_Date


In [154]:
conn.execute("ALTER TABLE dirt_cafe_sales RENAME COLUMN \"Transaction ID\" TO Transaction_ID;")


In [155]:
conn.execute("ALTER TABLE dirt_cafe_sales RENAME COLUMN \"Price Per Unit\" TO Price_Per_Unit;")

In [156]:
conn.execute("ALTER TABLE dirt_cafe_sales RENAME COLUMN \"Total Spent\" TO Total_Spent;")

In [157]:
conn.execute("ALTER TABLE dirt_cafe_sales RENAME COLUMN \"Payment Method\" TO Payment_Method;")

In [158]:
conn.execute("ALTER TABLE dirt_cafe_sales RENAME COLUMN \"Transaction Date\" TO Transaction_Date;")

In [159]:
#Viewing our table with the new column names
pd.read_sql("SELECT * FROM dirt_cafe_sales LIMIT 5;",conn)

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,9/8/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,5/16/2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,7/19/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,4/27/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,6/11/2023


In [160]:
#Investigating if there any duplicates on transaction_ID as this can be used as primary key
pd.read_sql("SELECT COUNT(DISTINCT Transaction_ID) FROM dirt_cafe_sales;",conn)

,COUNT(DISTINCT Transaction_ID)
0,10000


In [161]:
#Cleaning Payment Method
#Replacing all blanks and ERROR with UNKNOWN

conn.execute("UPDATE dirt_cafe_sales SET Payment_Method = 'UNKNOWN' WHERE Payment_Method = 'ERROR' OR Payment_Method IS NULL;")

In [162]:
#Cleaning Item column
#Replacing all blanks and ERROR with UNKNOWN

conn.execute("UPDATE dirt_cafe_sales SET Item = 'UNKNOWN' WHERE Item = 'ERROR' OR Item IS NULL;")

In [163]:
#Cleaning Item Location
#Replacing all blanks and ERROR with UNKNOWN
conn.execute("UPDATE dirt_cafe_sales SET Location = 'UNKNOWN' WHERE Location = 'ERROR' OR Location IS NULL;")

In [164]:
#Replacing any blanks, error or unknown with zero
#This will
conn.execute("""
UPDATE dirt_cafe_sales
SET Quantity = CASE
    WHEN Quantity IS NULL OR TRIM(Quantity) = '' OR Quantity = 'ERROR' OR Quantity = 'UNKNOWN' THEN 0
    ELSE Quantity
END,
Price_Per_Unit = CASE
    WHEN Price_Per_Unit IS NULL OR TRIM(Price_Per_Unit) = '' OR Price_Per_Unit = 'ERROR' OR Price_Per_Unit = 'UNKNOWN' THEN 0
    ELSE Price_Per_Unit
END,
 Total_Spent = CASE
    WHEN Total_Spent IS NULL OR TRIM(Total_Spent) = '' OR Total_Spent = 'ERROR' OR Total_Spent = 'UNKNOWN' THEN 0
    ELSE Total_Spent
END;
""")

In [165]:
#Cleaning transaction_date column
conn.execute("""
UPDATE dirt_cafe_sales
SET Transaction_Date = NULL
WHERE Transaction_Date = 'ERROR'
 OR Transaction_Date = 'UNKOWN'
 OR Transaction_Date = 'NULL';""")



In [166]:
from datetime import date
#Ensuring each column has the right data type
pd.read_sql("""
SELECT
    CAST(Quantity AS INTEGER) AS Quantity,
    CAST(Price_Per_Unit AS REAL) AS Price_Per_Unit,
    CAST(Total_Spent AS REAL) AS Total_Spent,
    CAST(Transaction_ID AS TEXT) AS Transaction_ID,
    CAST(Payment_Method AS TEXT) AS Payment_Method,
    CAST(Location AS TEXT) AS Location,
    Transaction_Date
FROM dirt_cafe_sales
LIMIT 5;
""", conn)

,Quantity,Price_Per_Unit,Total_Spent,Transaction_ID,Payment_Method,Location,Transaction_Date
0,2,2.0,4.0,TXN_1961373,Credit Card,Takeaway,9/8/2023
1,4,3.0,12.0,TXN_4977031,Cash,In-store,5/16/2023
2,4,1.0,0.0,TXN_4271903,Credit Card,In-store,7/19/2023
3,2,5.0,10.0,TXN_7034554,UNKNOWN,UNKNOWN,4/27/2023
4,2,2.0,4.0,TXN_3160411,Digital Wallet,In-store,6/11/2023


In [167]:
#Renaming my cleaned table
conn.execute("""
ALTER TABLE dirt_cafe_sales
RENAME TO clean_cafe_sales;
""")

In [168]:
pd.read_sql("""
SELECT * FROM clean_cafe_sales
LIMIT 6;
""",conn)

,Transaction_ID,Item,Quantity,Price_Per_Unit,Total_Spent,Payment_Method,Location,Transaction_Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,9/8/2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,5/16/2023
2,TXN_4271903,Cookie,4,1,0,Credit Card,In-store,7/19/2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,4/27/2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,6/11/2023
5,TXN_2602893,Smoothie,5,4,20,Credit Card,UNKNOWN,3/31/2023


In [169]:
#Let's see the top 5 items theat are mostly pusrchased
pd.read_sql("""
SELECT Item, SUM(Quantity) AS Total_Quantity FROM clean_cafe_sales
GROUP BY Item
ORDER BY Total_Quantity DESC
LIMIT 6;
""",conn )

,Item,Total_Quantity
0,Juice,3373
1,Coffee,3368
2,Cake,3329
3,Salad,3310
4,Sandwich,3245
5,Smoothie,3221


In [170]:
#I'm curios to know preference location of my buyers
pd.read_sql("""
SELECT Location, SUM(Quantity) AS Total_Quantity FROM clean_cafe_sales
GROUP BY Location
ORDER BY Total_Quantity DESC;
""",conn)

,Location,Total_Quantity
0,UNKNOWN,11392
1,In-store,8722
2,Takeaway,8720


In [171]:
#Let's look at product vs the revenue they generate
pd.read_sql("""
SELECT Item, SUM(Quantity) AS Total_Quality, Price_Per_Unit, SUM(Quantity * Price_Per_Unit) AS Cost FROM clean_cafe_sales
GROUP BY Item
ORDER BY Cost DESC;
""", conn)

,Item,Total_Quality,Price_Per_Unit,Cost
0,Salad,3310,5,15600.0
1,Sandwich,3245,4,12296.0
2,Smoothie,3221,4,12132.0
3,Juice,3373,3,9561.0
4,Cake,3329,3,9540.0
5,UNKNOWN,2744,3,7596.5
6,Coffee,3368,2,6424.0
7,Tea,3154,1.5,4431.0
8,Cookie,3090,1,2898.0


In [178]:
#Preferred payment method
pd.read_sql("""
SELECT SUM(Quantity) AS Total_Quantity, Payment_Method FROM clean_cafe_sales
GROUP BY Payment_Method
ORDER BY Total_Quantity ASC;
""",conn)

,Total_Quantity,Payment_Method
0,6533,Cash
1,6554,Credit Card
2,6647,Digital Wallet
3,9100,UNKNOWN
